# Investigacion final: weighted XGBoost por jets

Entrena variantes por jet usando `sample_weight`, compara politicas de threshold, exporta bundle y genera submission.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_invest_dir(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        direct = candidate / "01_weighted_xgboost_by_jets.py"
        nested = candidate / "14.invest" / "01_weighted_xgboost_by_jets.py"
        if direct.exists():
            return candidate
        if nested.exists():
            return candidate / "14.invest"
    raise FileNotFoundError("No se pudo ubicar 14.invest/01_weighted_xgboost_by_jets.py")


invest_dir = find_invest_dir()
project_root = invest_dir.parent
if str(invest_dir) not in sys.path:
    sys.path.insert(0, str(invest_dir))

import importlib.util

module_path = invest_dir / "01_weighted_xgboost_by_jets.py"
spec = importlib.util.spec_from_file_location("weighted_xgboost_by_jets", module_path)
weighted_module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(weighted_module)
run_investigation = weighted_module.run_investigation

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
print(f"invest_dir={invest_dir}")

invest_dir=/home/santig14/fing/taa/1_TallerAA/14.invest


## Entrenar variantes ponderadas

In [2]:
outputs = run_investigation()

print("Mejores variantes por jet")
display(outputs["best_by_jet"].round(6))

print("Politicas de threshold")
display(outputs["policy_comparison"].round(6))

jet=0 best_variant=sample_weight_count_spw_delta1 ams=0.764000 auc=0.888487


jet=1 best_variant=unweighted_shared_baseline ams=0.545509 auc=0.890798


jet=2 best_variant=sample_weight_count_spw_delta1 ams=0.624178 auc=0.918301


jet=3 best_variant=sample_weight_only_delta1 ams=0.258531 auc=0.884963


Mejores variantes por jet


,jet_num,variant,use_sample_weight,params_json,fit_seconds,train_rows,valid_rows,feature_count,threshold,ams,signal_weight,background_weight,predicted_signal_count,accuracy,precision,recall,f1,auc
0,0,sample_weight_count_spw_delta1,True,"{""colsample_bytree"": 0.9, ""eval_metric"": ""logl...",2.685585,89987,9926,78,0.086209,0.764000,14.896153,365.222468,1033,0.823393,0.862536,0.356115,0.504102,0.888487
1,1,unweighted_shared_baseline,False,"{""colsample_bytree"": 0.9, ""eval_metric"": ""logl...",3.046091,69731,7813,74,0.916754,0.545509,4.053031,43.867540,745,0.726482,0.944966,0.251429,0.397179,0.890798
2,2,sample_weight_count_spw_delta1,True,"{""colsample_bytree"": 0.9, ""eval_metric"": ""logl...",1.576625,45349,5030,67,0.054357,0.624178,4.411883,38.511734,1647,0.774354,0.940498,0.598995,0.731869,0.918301
3,3,sample_weight_only_delta1,True,"{""colsample_bytree"": 0.9, ""eval_metric"": ""logl...",0.955802,19933,2231,67,0.025269,0.258531,1.653627,30.364321,500,0.836396,0.814000,0.599411,0.690416,0.884963


Politicas de threshold


,policy,accuracy,precision,recall,f1,auc,ams,signal_weight,background_weight,predicted_signal_count,thresholds
0,joint_thresholds_from_independent,0.77220,0.917442,0.368390,0.525693,0.819138,1.130993,18.600245,254.339273,3440,"{""0"": 0.20778033137321472, ""1"": 0.916754007339..."
1,joint_thresholds_from_global,0.79624,0.896190,0.458504,0.606641,0.819138,1.128131,27.397589,570.739266,4383,"{""0"": 0.0862087681889534, ""1"": 0.8857419490814..."
2,independent_threshold_by_jet,0.78440,0.904713,0.414497,0.568524,0.819138,1.122927,25.014693,477.966064,3925,"{""0"": 0.0862087681889534, ""1"": 0.9167540073394..."
3,global_single_threshold,0.69136,0.956056,0.104120,0.187789,0.819138,0.587597,4.369806,43.867540,933,0.916754007339


## Comparacion contra resultados previos

In [3]:
weighted_best = outputs["policy_comparison"].iloc[0]

plan12 = pd.read_csv(project_root / "12.plan" / "results" / "08_final_comparison.csv")
plan12_best = plan12.loc[plan12["dataset"] == "xgboost_i3_tree_safe_mix_by_jets_final"].iloc[0]

comb13 = pd.read_csv(project_root / "13.comb" / "results" / "01_blend_sweep_metrics.csv")
comb13_best = comb13.iloc[0]

comparison_df = pd.DataFrame([
    {
        "model": "12.plan xgboost por jets final",
        "policy": str(plan12_best["policy"]),
        "auc": float(plan12_best["auc"]),
        "ams": float(plan12_best["ams"]),
        "precision": float(plan12_best["precision"]),
        "recall": float(plan12_best["recall"]),
    },
    {
        "model": "13.comb i3 + v2",
        "policy": str(comb13_best["blend_key"]),
        "auc": float(comb13_best["auc"]),
        "ams": float(comb13_best["ams"]),
        "precision": float(comb13_best["precision"]),
        "recall": float(comb13_best["recall"]),
    },
    {
        "model": "14.invest weighted xgboost por jets",
        "policy": str(weighted_best["policy"]),
        "auc": float(weighted_best["auc"]),
        "ams": float(weighted_best["ams"]),
        "precision": float(weighted_best["precision"]),
        "recall": float(weighted_best["recall"]),
    },
])

comparison_path = invest_dir / "results" / "02_final_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
display(comparison_df.round(6))
print(f"Comparacion guardada en: {comparison_path}")

,model,policy,auc,ams,precision,recall
0,12.plan xgboost por jets final,joint_thresholds_from_independent,0.903617,1.147119,0.928863,0.397806
1,13.comb i3 + v2,percentile_rank_i3_0.30_v2_0.70,0.910106,1.118636,0.943820,0.362787
2,14.invest weighted xgboost por jets,joint_thresholds_from_independent,0.819138,1.130993,0.917442,0.368390


Comparacion guardada en: /home/santig14/fing/taa/1_TallerAA/14.invest/results/02_final_comparison.csv


## Archivos finales

In [4]:
metadata = outputs["metadata"]
print(f"Submission: {metadata['submission_path']}")
print(f"Bundle: {metadata['bundle_dir']}")
print("Validacion submission:")
display(pd.DataFrame([metadata["submission_validation"]]))

Submission: /home/santig14/fing/taa/1_TallerAA/14.invest/results/higgs_submission_weighted_xgboost_by_jets.csv
Bundle: /home/santig14/fing/taa/1_TallerAA/14.invest/models/weighted_xgboost_by_jets_bundle
Validacion submission:


,rows,rank_min,rank_max,signal_count,background_count
0,550000,1,550000,74431,475569
